In [ ]:
import numpy as np
import pandas as pd
import warnings
import re
import pickle
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import  Embedding, LSTM, GRU, Dense, Dropout

In [ ]:
data = pd.read_csv("/content/DataScience QA.csv")
data.head()

,Question,Answer
0,What is the purpose of feature engineering in ...,"Feature engineering involves selecting, transf..."
1,What are some common techniques for feature en...,Common techniques include one-hot encoding for...
2,What is the difference between batch processin...,Batch processing involves processing data in l...
3,What is natural language processing (NLP)?,Natural language processing is a field of arti...
4,What is sentiment analysis?,Sentiment analysis is the task of automaticall...


In [ ]:
data.isnull().sum()

,0
Question,0
Answer,0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Question  158 non-null    object
 1   Answer    158 non-null    object
dtypes: object(2)
memory usage: 2.6+ KB


In [ ]:
data.shape

(158, 2)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

In [ ]:
data['Question'] = data['Question'].apply(clean_text)
data['Answer'] = data['Answer'].apply(clean_text)

In [ ]:
data['QA_combined'] = data['Question'] + " " + data['Answer']

In [ ]:
max_words = 10000
max_len = 50
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(data['QA_combined'])
X = tokenizer.texts_to_sequences(data['QA_combined'])
X = pad_sequences(X, maxlen=max_len)

In [ ]:
data['Score'] = data['Answer'].apply(lambda x: len(x.split()))

In [ ]:
data['answer_len'] = data['Answer'].apply(lambda x: len(x.split()))
def categorize(length):
    if length < 10:
        return 0
    elif length < 30:
        return 1
    else:
        return 2

In [ ]:
data['label'] = data['answer_len'].apply(categorize)

In [ ]:
y = data['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(128))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(3, activation='softmax'))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 207ms/step - accuracy: 0.6153 - loss: 1.0844 - val_accuracy: 0.7188 - val_loss: 1.0335
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.6226 - loss: 1.0145 - val_accuracy: 0.7188 - val_loss: 0.8603
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.6216 - loss: 0.8303 - val_accuracy: 0.7500 - val_loss: 0.5193
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.7086 - loss: 0.5287 - val_accuracy: 0.7500 - val_loss: 0.4906
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.7307 - loss: 0.4705 - val_accuracy: 0.8125 - val_loss: 0.4243
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.9254 - loss: 0.3721 - val_accuracy: 0.8750 - val_loss: 0.3898
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.9592 - loss: 0.3367 - val_accuracy: 0.8750 - val_loss: 0.3522
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.9947 - loss: 0.2649 - val_accuracy: 0.9062 - val_loss:

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.6562 - loss: 0.9471
Test Loss: 0.947054922580719
Test Accuracy: 0.65625


In [ ]:
def predict_answer_quality(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(padded)
    label = np.argmax(pred)
    confidence = np.max(pred)
    labels = ["Short Answer", "Medium Answer", "Detailed Answer"]
    return labels[label], confidence

In [ ]:
predict_answer_quality(
"What is machine learning? Machine learning is a subset of AI that allows computers to learn from data."
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step


('Medium Answer', np.float32(0.91099465))

In [ ]:
model.save("emotion_model.keras")

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
preprocessor = {
    "tokenizer": tokenizer,
    "label_encoder": label_encoder,
    "max_len": max_len
}

with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)